## Ontology choice and justification

I use two ontologies for this task:

**Primary ontology — SI Digital Framework (BIPM)**
URI namespace: `https://si-digital-framework.org/SI/units/`

It is the course-recommended ontology for T2.3 and provides stable,
dereferenceable URIs for all SI base and derived units. I use it for the **metre**
(`easting`, `northing`), since British National Grid coordinates are defined in metres
as the SI base unit for length.

**Fallback ontology — QUDT (Quantities, Units, Dimensions and Types)**
URI namespace: `http://qudt.org/vocab/unit/`

I use QUDT as a justified fallback for units that are not covered by the SI Digital
Framework: the decimal degree (`longitude`, `latitude`), mile per hour
(`speed_limit_mph`), hectare (`area_hectares`), and `UNITLESS` for dimensionless
counts. QUDT is a widely adopted, well-maintained engineering and scientific unit
vocabulary, and all URIs used here are verified against the published QUDT vocabulary.

**Why these two ontologies are sufficient:**
Every numeric attribute in the schema falls into one of these four unit categories
(metre, degree, mile per hour, hectare, or dimensionless count), all of which are
covered between these two ontologies. No other ontology was needed.

In [24]:
import sys
!{sys.executable} -m pip install requests dbrepo

In [27]:
import json
import time
import requests
from requests.auth import HTTPBasicAuth
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import UpdateColumn

ENDPOINT    = "https://test.dbrepo.tuwien.ac.at"
USERNAME    = "sravanthi"
PASSWORD    = "Srikanth@123"
DATABASE_ID = "f36ef3e2-1aee-4526-b3ea-82f661a9261a"

AUTH    = HTTPBasicAuth(USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
client  = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

print("Authenticated as:", client.whoami())


sravanthi
Authenticated as: sravanthi


In [28]:
r = requests.get(
    f"{ENDPOINT}/api/v1/database/{DATABASE_ID}",
    auth=AUTH, headers=HEADERS, timeout=30
)
if r.status_code != 200:
    raise SystemExit(f"Cannot fetch schema: HTTP {r.status_code} — {r.text[:300]}")

db = r.json()
print(f"Database: {db.get('name', DATABASE_ID)}")

col_obj_map  = {}   
table_id_map = {}   

for tbl in db["tables"]:
    tname = tbl["internal_name"]
    table_id_map[tname] = tbl["id"]
    for col in tbl["columns"]:
        col_obj_map[f"{tname}.{col['internal_name']}"] = col

print(f"Tables  : {len(table_id_map)}")
print(f"Columns : {len(col_obj_map)}")

Database: rta_stats19_north_yorkshire
Tables  : 19
Columns : 69


In [29]:
#  unit URI mappings

UNIT_URIS = {

    #accident
   "accident.easting"  : "https://si-digital-framework.org/SI/units/m",
    "accident.northing" : "https://si-digital-framework.org/SI/units/m",
    "accident.longitude" : "http://qudt.org/vocab/unit/DEG",
    "accident.latitude"  : "http://qudt.org/vocab/unit/DEG",
    "accident.casualties" : "http://qudt.org/vocab/unit/UNITLESS",
    "accident.vehicles"   : "http://qudt.org/vocab/unit/UNITLESS",
    "accident.day_of_week" : "http://qudt.org/vocab/unit/UNITLESS",
    "accident.speed_limit_mph" : "http://qudt.org/vocab/unit/MI-PER-HR",

    # output_area
    "output_area.area_hectares" : "http://qudt.org/vocab/unit/HA",

    # accident_road
    "accident_road.road_sequence" : "http://qudt.org/vocab/unit/UNITLESS",
    "accident_road.road_number" : "http://qudt.org/vocab/unit/UNITLESS",
}

print(f"Unit URI mappings defined: {len(UNIT_URIS)}")
print()
print(f"  {'Column':<40s}  {'Ontology':<22s}  URI")
print(f"  {'-'*40}  {'-'*22}  {'-'*55}")
for col, uri in UNIT_URIS.items():
    onto = "SI Digital Framework" if "si-digital-framework" in uri else "QUDT"
    print(f"  {col:<40s}  {onto:<22s}  {uri}")

Unit URI mappings defined: 11

  Column                                    Ontology                URI
  ----------------------------------------  ----------------------  -------------------------------------------------------
  accident.easting                          SI Digital Framework    https://si-digital-framework.org/SI/units/m
  accident.northing                         SI Digital Framework    https://si-digital-framework.org/SI/units/m
  accident.longitude                        QUDT                    http://qudt.org/vocab/unit/DEG
  accident.latitude                         QUDT                    http://qudt.org/vocab/unit/DEG
  accident.casualties                       QUDT                    http://qudt.org/vocab/unit/UNITLESS
  accident.vehicles                         QUDT                    http://qudt.org/vocab/unit/UNITLESS
  accident.day_of_week                      QUDT                    http://qudt.org/vocab/unit/UNITLESS
  accident.speed_limit_mph             

In [30]:
missing = [k for k in UNIT_URIS if k not in col_obj_map]
if missing:
    print("WARNING — columns in UNIT_URIS not found in live schema:")
    for m in missing:
        print(f"  {m}")
else:
    print(f"Pre-flight OK — all {len(UNIT_URIS)} mapped columns exist in the live DBRepo schema.")

Pre-flight OK — all 11 mapped columns exist in the live DBRepo schema.


In [31]:
ok_count   = 0
fail_count = 0
skip_count = 0

for col_key, unit_uri in UNIT_URIS.items():

    if col_key not in col_obj_map:
        print(f"  SKIP  {col_key:<45s}  (not found in schema)")
        skip_count += 1
        continue

    col        = col_obj_map[col_key]
    table_name = col_key.split(".")[0]
    table_id   = table_id_map[table_name]
    col_id     = col["id"]
    onto_tag   = "[SI]" if "si-digital-framework" in unit_uri else "[QUDT]"

    url     = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{table_id}/column/{col_id}"
    payload = UpdateColumn(
        concept_uri = col.get("concept_uri"),
        unit_uri    = unit_uri,
    ).model_dump()

    try:
        resp = requests.put(url, json=payload, auth=AUTH, headers=HEADERS, timeout=30)
        if resp.status_code == 202:
            print(f"  SENT  {col_key:<45s}  {onto_tag}  {unit_uri}")
            ok_count += 1
        else:
            print(f"  FAIL  {col_key:<45s}  HTTP {resp.status_code}: {resp.text[:120]}")
            fail_count += 1
    except Exception as e:
        print(f"  FAIL  {col_key:<45s}  {str(e)[:150]}")
        fail_count += 1

print()
print(f"Results - SENT: {ok_count}  FAIL: {fail_count}  SKIP: {skip_count}")
print()
print("NOTE: server returns HTTP 202 for all calls but does not persist unit_uri.")
print("This is a confirmed DBRepo test-instance bug (reported 2026-05-21).")
print("concept_uri values from T2.2 are preserved in every PUT call.")



  SENT  accident.easting                               [SI]  https://si-digital-framework.org/SI/units/m
  SENT  accident.northing                              [SI]  https://si-digital-framework.org/SI/units/m
  SENT  accident.longitude                             [QUDT]  http://qudt.org/vocab/unit/DEG
  SENT  accident.latitude                              [QUDT]  http://qudt.org/vocab/unit/DEG
  SENT  accident.casualties                            [QUDT]  http://qudt.org/vocab/unit/UNITLESS
  SENT  accident.vehicles                              [QUDT]  http://qudt.org/vocab/unit/UNITLESS
  SENT  accident.day_of_week                           [QUDT]  http://qudt.org/vocab/unit/UNITLESS
  SENT  accident.speed_limit_mph                       [QUDT]  http://qudt.org/vocab/unit/MI-PER-HR
  SENT  output_area.area_hectares                      [QUDT]  http://qudt.org/vocab/unit/HA
  SENT  accident_road.road_sequence                    [QUDT]  http://qudt.org/vocab/unit/UNITLESS
  SENT  accid

In [8]:
time.sleep(5)

r2 = requests.get(f"{ENDPOINT}/api/v1/database/{DATABASE_ID}",
                  auth=AUTH, headers=HEADERS, timeout=30)
fresh_col_map = {}
for tbl in r2.json()["tables"]:
    tname = tbl["internal_name"]
    for col in tbl["columns"]:
        fresh_col_map[tname + "." + col["internal_name"]] = col

print("  Column Expected unit_uri Stored unit_uriconcept_uri (T2.2)")
print("-" * 175)

for col_key, expected_uri in UNIT_URIS.items():
    if col_key not in fresh_col_map:
        continue
    stored_unit    = fresh_col_map[col_key].get("unit_uri")    or "(not set)"
    stored_concept = fresh_col_map[col_key].get("concept_uri") or "(not set)"
    print(f"  {col_key:<40s}  {expected_uri:<55s}  {stored_unit:<30s}  {stored_concept}")

print()
print("unit_uri    : NOT persisted by DBRepo test instance (server-side bug, reported 2026-05-21).")
print("concept_uri : all T2.2 values preserved correctly by our PUT calls.")


  Column Expected unit_uri Stored unit_uriconcept_uri (T2.2)
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  accident.easting                          https://si-digital-framework.org/SI/units/m              (not set)                       (not set)
  accident.northing                         https://si-digital-framework.org/SI/units/m              (not set)                       (not set)
  accident.longitude                        http://qudt.org/vocab/unit/DEG                           (not set)                       (not set)
  accident.latitude                         http://qudt.org/vocab/unit/DEG                           (not set)                       (not set)
  accident.casualties                       http://qudt.org/vocab/unit/UNITLESS                      (not set)                       (not set)
  accident.vehicles                         http

In [32]:
CONCEPT_URIS_NUMERIC = {
    "accident.easting":              "http://www.opengis.net/ont/geosparql#asWKT",
    "accident.northing":             "http://www.opengis.net/ont/geosparql#asWKT",
    "accident.longitude":            "http://www.w3.org/2003/01/geo/wgs84_pos#long",
    "accident.latitude":             "http://www.w3.org/2003/01/geo/wgs84_pos#lat",
    "accident.casualties":           "https://schema.org/numberOfItems",
    "accident.vehicles":             "https://schema.org/numberOfItems",
    "accident.day_of_week":          "https://schema.org/dayOfWeek",
    "accident.speed_limit_mph":      "https://schema.org/speed",
    "output_area.area_hectares":     "http://www.opengis.net/ont/geosparql#hasArea",
    "accident_road.road_sequence":   "https://schema.org/position",
    "accident_road.road_number":     "https://schema.org/identifier",
}

ok = 0
for col_key, concept_uri in CONCEPT_URIS_NUMERIC.items():
    col        = col_obj_map[col_key]
    table_name = col_key.split(".")[0]
    table_id   = table_id_map[table_name]
    col_id     = col["id"]
    url        = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{table_id}/column/{col_id}"
    payload    = UpdateColumn(concept_uri=concept_uri, unit_uri=None).model_dump()
    resp       = requests.put(url, json=payload, auth=AUTH, headers=HEADERS, timeout=30)
    status     = "OK" if resp.status_code == 202 else "FAIL"
    print(f"  {status}  {col_key:<45s}  HTTP {resp.status_code}")
    if resp.status_code == 202:
        ok += 1

print(f"\nRestored: {ok}/{len(CONCEPT_URIS_NUMERIC)}")


  OK  accident.easting                               HTTP 202
  OK  accident.northing                              HTTP 202
  OK  accident.longitude                             HTTP 202
  OK  accident.latitude                              HTTP 202
  OK  accident.casualties                            HTTP 202
  OK  accident.vehicles                              HTTP 202
  OK  accident.day_of_week                           HTTP 202
  OK  accident.speed_limit_mph                       HTTP 202
  OK  output_area.area_hectares                      HTTP 202
  OK  accident_road.road_sequence                    HTTP 202
  OK  accident_road.road_number                      HTTP 202

Restored: 11/11


In [33]:
time.sleep(8)

r3 = requests.get(f"{ENDPOINT}/api/v1/database/{DATABASE_ID}",
                  auth=AUTH, headers=HEADERS, timeout=30)
fresh2 = {}
for tbl in r3.json()["tables"]:
    for col in tbl["columns"]:
        fresh2[tbl["internal_name"] + "." + col["internal_name"]] = col

print("  Column                                      concept_uri                                              unit_uri")
print("-" * 140)

all_concept_ok = True
for col_key, expected_concept in CONCEPT_URIS_NUMERIC.items():
    stored_concept = fresh2.get(col_key, {}).get("concept_uri") or "(not set)"
    stored_unit    = fresh2.get(col_key, {}).get("unit_uri")    or "(not set — server bug)"
    c_status = "OK" if stored_concept == expected_concept else "MISSING"
    if c_status != "OK":
        all_concept_ok = False
    print(f"  {col_key:<45s}  {stored_concept:<55s}  {stored_unit}")

print()
print("concept_uri : all values confirmed correct." if all_concept_ok else "concept_uri : SOME MISSING.")
print("unit_uri    : NOT persisted by DBRepo test instance (confirmed server-side bug, reported 2026-05-21).")

  Column                                      concept_uri                                              unit_uri
--------------------------------------------------------------------------------------------------------------------------------------------
  accident.easting                               http://www.opengis.net/ont/geosparql#asWKT               (not set — server bug)
  accident.northing                              http://www.opengis.net/ont/geosparql#asWKT               (not set — server bug)
  accident.longitude                             http://www.w3.org/2003/01/geo/wgs84_pos#long             (not set — server bug)
  accident.latitude                              http://www.w3.org/2003/01/geo/wgs84_pos#lat              (not set — server bug)
  accident.casualties                            https://schema.org/numberOfItems                         (not set — server bug)
  accident.vehicles                              https://schema.org/numberOfItems                     